# Single-Dataset SOTA Model — Train on OAI, Test on Others

This notebook trains a strong model on a single dataset (OAI) and evaluates it on the other three datasets in the same run. The strategy is single-source: maximise accuracy on one well-curated dataset, then report honestly how that model transfers to unseen sources.

OAI is chosen as the training source because it is the largest radiologist-graded dataset and the standard benchmark in the literature, so its in-domain accuracy is directly comparable to published results. The model uses the full accuracy configuration: an ordinal (CORN) head, full fine-tuning, and test-time augmentation.

The external datasets (Mendeley, MRKR, NHANES) are passed to the training routine as a combined test set, so their metrics are returned directly and no separate checkpoint step is required.

## Setup

Loads the shared configuration, training library, packed image array, and manifest. A GPU is required for training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, os, json, time, math
sys.path.insert(0, '/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import numpy as np, pandas as pd
import torch
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
from pathlib import Path
from sklearn.metrics import accuracy_score, cohen_kappa_score, roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda': raise RuntimeError('GPU required for training.')

PROJECT = Path('/content/drive/MyDrive/Master Thesis')
SS_ROOT = PROJECT/'scope3_single'; SS_CKPT = SS_ROOT/'checkpoints'; SS_RES = SS_ROOT/'results'
for d in (SS_ROOT, SS_CKPT, SS_RES): d.mkdir(parents=True, exist_ok=True)
print('Output folder:', SS_ROOT)

## Training source and configuration

The model is trained only on OAI's training split; the other three datasets are reserved entirely for external testing. The configuration is the full accuracy setup. The multi-dataset mechanisms (curriculum, domain-adversarial) are switched off, since they have no role when training on a single source.

In [ ]:
TRAIN_DS = 'oai'                              # train on this dataset only
EXTERNAL = ['mendeley', 'mrkr', 'nhanes3']    # test on these, never trained on

config_flags = {
    'sampler': True, 'noise_loss': True, 'curriculum': False, 'domain_adv': False,
    'hierarchical': True, 'use_quality': True, 'soft_alpha': TM.MAX_SOFT_ALPHA,
    'freeze_stages': TM.MAX_FREEZE_STAGES, 'grl_lambda_max': 0.0,
}
print('Train on:', TRAIN_DS, '| Test on:', EXTERNAL)

## Build the single-source splits

The data is restricted to OAI, then split into training and validation **by patient**, so that all images from one patient (both knees, every visit) stay on the same side. A random row-level split would place the same patient on both sides and leak. The other three datasets are loaded separately as external test sets.

In [ ]:
manifest = TM.prepare_local_data()
manifest['patient_id'] = manifest['dataset'].astype(str) + '::' + \
    manifest['filename'].astype(str).str.extract(r'(\d{5,})')[0].fillna('na')

# Restrict to OAI and split BY PATIENT (group-aware), not by row.
src = manifest[manifest['dataset'] == TRAIN_DS].reset_index(drop=True)
patients = src['patient_id'].drop_duplicates().sample(frac=1.0, random_state=0).tolist()
n_val = max(1, int(round(0.15 * len(patients))))
val_patients = set(patients[:n_val])
va = src[src['patient_id'].isin(val_patients)].reset_index(drop=True)
tr = src[~src['patient_id'].isin(val_patients)].reset_index(drop=True)
assert len(set(tr['patient_id']) & set(va['patient_id'])) == 0, 'leak between train/val'

externals = {ds: manifest[manifest['dataset'] == ds].reset_index(drop=True) for ds in EXTERNAL}
print(f'OAI patients: {len(patients)}  ->  train {len(tr):,} imgs / val {len(va):,} imgs (split by patient)')
for ds, df in externals.items():
    print(f'  external {ds}: {len(df):,}')

## Train on OAI, evaluate on the external datasets

The three external datasets are combined into one tagged test set and passed to the training routine, so the returned metrics include external performance directly — no checkpoint reload is needed afterward. A unique run name guarantees the routine does not skip the run as already complete. In-domain accuracy is measured on the OAI validation split; external accuracy is measured on the combined Mendeley/MRKR/NHANES set.

In [4]:
# Combine the three external datasets into one tagged test set.
ext_df = pd.concat([externals[d].assign(_ext_ds=d) for d in EXTERNAL], ignore_index=True)
print(f'Combined external test set: {len(ext_df):,} images from {EXTERNAL}')

# Unique run name so the routine cannot skip it; pass force/overwrite if supported.
import inspect
RUN_NAME = f"single_{TRAIN_DS}_{time.strftime('%H%M%S')}"
kwargs = dict(seed=0, log_fn=print)
if 'force' in inspect.signature(TM.run_training).parameters: kwargs['force'] = True
if 'overwrite' in inspect.signature(TM.run_training).parameters: kwargs['overwrite'] = True

# Train on OAI (tr), select on OAI val (va), evaluate on the combined externals (ext_df).
res = TM.run_training(RUN_NAME, tr, va, ext_df, config_flags, **kwargs)
assert isinstance(res, dict) and 'internal_acc5' in res, 'Training did not return metrics.'

in_acc  = res['internal_acc5']      # OAI in-domain (validation)
ext_acc = res.get('external_acc5')  # pooled accuracy on the 3 external datasets
assert in_acc > 0.45, ('In-domain accuracy implausibly low (%.3f) — model likely not trained.' % in_acc)

print()
print('OAI in-domain accuracy:      %.1f%%' % (in_acc*100))
print('External (pooled) accuracy:  %.1f%%' % (ext_acc*100))
print('Gap (in-domain - external):  %+.1f pp' % ((in_acc-ext_acc)*100))

Combined external test set: 53,011 images from ['mendeley', 'mrkr', 'nhanes3']
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 218MB/s]


  [single_oai_211639] ep1/18 loss=1.170 tr=0.234 val=0.400 w1=0.593 qwk=0.004 gap=-0.166 (113s)
  [single_oai_211639] ep2/18 loss=1.090 tr=0.280 val=0.449 w1=0.706 qwk=0.373 gap=-0.170 (89s)
  [single_oai_211639] ep3/18 loss=0.963 tr=0.422 val=0.519 w1=0.815 qwk=0.601 gap=-0.097 (89s)
  [single_oai_211639] ep4/18 loss=0.883 tr=0.509 val=0.541 w1=0.843 qwk=0.651 gap=-0.032 (89s)
  [single_oai_211639] ep5/18 loss=0.835 tr=0.545 val=0.554 w1=0.850 qwk=0.665 gap=-0.009 (89s)
  [single_oai_211639] ep6/18 loss=0.819 tr=0.550 val=0.579 w1=0.862 qwk=0.701 gap=-0.028 (89s)
  [single_oai_211639] ep7/18 loss=0.785 tr=0.582 val=0.575 w1=0.864 qwk=0.695 gap=+0.007 (89s)
  [single_oai_211639] ep8/18 loss=0.785 tr=0.576 val=0.593 w1=0.876 qwk=0.734 gap=-0.016 (89s)
  [single_oai_211639] ep9/18 loss=0.755 tr=0.596 val=0.606 w1=0.885 qwk=0.745 gap=-0.010 (89s)
  [single_oai_211639] ep10/18 loss=0.743 tr=0.607 val=0.606 w1=0.879 qwk=0.738 gap=+0.001 (89s)
  [single_oai_211639] ep11/18 loss=0.733 tr=0.62

## Summary

The headline of the single-source approach: a strong in-domain accuracy on OAI, and the accuracy retained on the unseen datasets. A gap between the two is expected and is the honest cost of training on a single source; it also motivates the multi-dataset approach studied separately. All figures are saved to a JSON file for the report.

In [5]:
summary = {
    'train_ds': TRAIN_DS,
    'external_datasets': EXTERNAL,
    'in_domain_acc': float(in_acc),
    'external_pooled_acc': float(ext_acc),
    'external_within1': float(res.get('external_within1', float('nan'))),
    'external_qwk': float(res.get('external_qwk', float('nan'))),
    'external_auc_oa': float(res.get('external_auc_oa', float('nan'))),
    'gap_pp': float((in_acc - ext_acc) * 100),
    'n_train': int(len(tr)), 'n_val': int(len(va)), 'n_external': int(len(ext_df)),
    'history': res.get('history'),
}
json.dump(summary, open(str(SS_RES/f'{RUN_NAME}_summary.json'), 'w'), indent=2)

print('='*52)
print(f'SINGLE-SOURCE MODEL (trained on {TRAIN_DS.upper()})')
print('='*52)
print(f'In-domain accuracy (OAI):        {in_acc*100:.1f}%')
print(f'External accuracy (3 datasets):  {ext_acc*100:.1f}%')
print(f'External within-1 accuracy:      {res.get("external_within1", float("nan"))*100:.1f}%')
print(f'External binary OA AUC:          {res.get("external_auc_oa", float("nan")):.3f}')
print(f'Gap (in-domain - external):      {(in_acc-ext_acc)*100:+.1f} pp')
print('='*52)
print('Saved -> ' + f'{RUN_NAME}_summary.json')

SINGLE-SOURCE MODEL (trained on OAI)
In-domain accuracy (OAI):        63.3%
External accuracy (3 datasets):  48.7%
External within-1 accuracy:      70.3%
External binary OA AUC:          0.808
Gap (in-domain - external):      +14.7 pp
Saved -> single_oai_211639_summary.json
